In [ ]:
import vibechecker as vc
import sounddevice
import yaml
import pickle
from datetime import datetime, tzinfo
import pandas as pd
import numpy as np
import plotly.express as px
from path import Path
import h5py
import time
import os
import glob
import queue

In [ ]:
# Launch GUI
app = vc.GUI()
app.serve()

In [ ]:
# test load all datasets
datasets = list(map(Path,glob.glob('DEVDATA/*.h5')))
samps = []
for dataset in datasets:
    if dataset.basename().startswith('pytest'):
        continue

    samps.append(vc.VibeSample.load(dataset))


In [ ]:
# Test units comparisson
dataset = Path('DEVDATA/rotorkit_1800rpm_2025-12-30_15-03-02.h5')
samp = vc.VibeSample.load(dataset) # type: ignore

In [ ]:
str(samp._timestamp)

In [ ]:

config = vc.AcquisitionSettings(len(samp.data), samp.samplerate, units='g', integrate=False)
vtime, vaccel = samp.get_accel(config)
df,peak,rms = samp.fft(config)
df.psd.max()

In [ ]:
samp

In [ ]:
devs = vc.VibeSensor.find()

print('ID\tNAME')
for s in devs:
    print(f'{s.device_id}:\t{s.model_name}')

In [ ]:
q = queue.Queue()
def callback(sample):
    q.put(sample)

config = vc.AcquisitionSettings(vc.SAMPLERATES[2], vc.BLOCKSIZES[-1])
sensor = devs[-1]

# Close any existing stream if present and active
try:
    stream.close() # type: ignore
except Exception:
    pass

stream = sensor.connect(config, callback)

stream.start()
time.sleep(1)
stream.stop()
stream.close()

while q.qsize():
    sample = q.get()
q.shutdown()

ch = 0
vs = vc.VibeSample(sample['status'],
                   sample['timestamp'],
                   config.samplerate,
                   config.units,
                   sample['data'][:,ch])

td = np.arange(vs.blocksize) / vs.samplerate
df = pd.DataFrame({'time': td,
                   'accel0': sample['data'][:,0],
                   'accel1': sample['data'][:,1],
                   'accelm': np.mean(sample['data'], axis=1)})
px.line(df, x='time', y=['accel0', 'accel1', 'accelm'])

In [ ]:
h5file = vs.save()
time.sleep(.1)
vs2 = vc.VibeSample.load(h5file)
if not (vs.data == vs2.data).all():
    print(vs.data - vs2.data)

In [ ]:
h5file = Path('DEVDATA/zrotor_test_save.h5')

# if h5file.exists():
#     os.remove(h5file)

vs.save(h5file)
time.sleep(.2)
vs2 = vc.VibeSample.load(h5file) # type: ignore


In [ ]:
vs.save()

In [ ]:
# Generate and visualize simulated data
dev = vc.VibeSensor.find()

if len(dev)>1:
    sensor = dev[1]
else:
    sensor = dev[0]

print(sensor)

config = vc.AcquisitionSettings(4096,8000)

config.ensure_maxfreq(1000)
config.ensure_binsize(2.0)

vibr = vc.DataCollector(sensor=sensor, config=config)
samp = vibr.collect_sample()
vis = vibr.visualize_init(samp)

vibr.disconnect_sensor()

In [ ]:
import dearpygui.dearpygui as dpg

fft_data_x = list(range(100))
fft_data_y = [i**2 for i in fft_data_x]

dpg.create_context()

def update_crosshair(sender, app_data):
    plot = dpg.get_plot_mouse_pos()
    x, y = plot[0], plot[1]

    # Check if mouse is within plot bounds (get_plot_mouse_pos returns extreme values when outside)
    if x > 1e6 or x < -1e6:
        if dpg.does_item_exist("v_line"):
            dpg.delete_item("v_line")
            dpg.delete_item("h_line")
        if dpg.does_item_exist("tooltip"):
            dpg.delete_item("tooltip")
        
        return

    # Find closest index
    idx = min(range(len(fft_data_x)), key=lambda i: abs(fft_data_x[i] - x))
    y_value = fft_data_y[idx]
    cross_width = (max(fft_data_x) - min(fft_data_x)) * 0.05

    # Update or create crosshair
    if dpg.does_item_exist("v_line"):
        dpg.configure_item("v_line", p1=[fft_data_x[idx], 0], p2=[fft_data_x[idx], max(fft_data_y)])
        dpg.configure_item("h_line", 
                           p1=[fft_data_x[idx]-cross_width, y_value],
                           p2=[fft_data_x[idx]+cross_width, y_value])
    else:
        dpg.draw_line([fft_data_x[idx], 0], [fft_data_x[idx], max(fft_data_y)], 
                      color=[255, 0, 0, 255], thickness=.1, parent="plot", tag="v_line")
        dpg.draw_line([fft_data_x[idx]-cross_width, y_value],
                      [fft_data_x[idx]+cross_width, y_value],
                      color=[255, 0, 0, 255], thickness=.1, parent="plot", tag="h_line")

    if dpg.does_item_exist("tooltip"):
        dpg.configure_item("tooltip", default_value=f"Value: {y_value}", pos=[fft_data_x[idx] + 10, y_value + 10])
    else:
        dpg.draw_text(pos=[[fft_data_x[idx] + 10, y_value + 10]], text=f"Value: {y_value}",
                      color=[255, 255, 0, 255], parent="plot", tag="tooltip")

# Mouse move handler
with dpg.handler_registry():
    dpg.add_mouse_move_handler(callback=update_crosshair)

with dpg.window(label="Plot Window"):
    with dpg.plot(label="FFT Plot", height=400, width=600, tag="plot"):
        dpg.add_plot_axis(dpg.mvXAxis, label="X")
        dpg.add_plot_axis(dpg.mvYAxis, label="Y")
        dpg.add_line_series(fft_data_x, fft_data_y, label="FFT Data", parent=dpg.last_item(), tag="fft_data")

dpg.create_viewport()
dpg.setup_dearpygui()
dpg.show_viewport()
dpg.start_dearpygui()
dpg.destroy_context()     